# Human vs AI Wikipedia: Matched Coherence Decay Comparison

Runs the corrected token-reveal pipeline on **AI-generated** Wikipedia articles
(produced by Claude Sonnet) matched by topic and language to the human articles.

**Languages**: English, Chinese, Turkish (3 unrelated families)

**Human results (already computed for zh, tr; new for en)**:
- Llama Chinese: α = -0.77, Turkish: α = -0.88
- Mistral Chinese: α = -0.82, Turkish: α = -0.79

**Key question**: Do AI articles on the same topics in the same languages show
a different exponent? If yes → the probe measures something real about text
structure, not its own bias.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Config ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    AI_DATA_DIR = Path('/content/drive/MyDrive/LRTIA/Data/wiki_multilingual_ai')
    HUMAN_DATA_DIR = Path('/content/drive/MyDrive/LRTIA/Data/wiki_multilingual')
    BASE_DIR = Path('/content/drive/MyDrive/LRTIA/Results/AI_vs_Human_wiki')
    HUMAN_RESULTS = Path('/content/drive/MyDrive/LRTIA/Results')
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    AI_DATA_DIR = Path('../data/wiki_multilingual_ai')
    HUMAN_DATA_DIR = Path('../data/wiki_multilingual')
    BASE_DIR = Path('../results/AI_vs_Human_wiki')
    HUMAN_RESULTS = Path('../results')
    BASE_DIR.mkdir(parents=True, exist_ok=True)

PROBE_MODELS = {
    'mistral': {
        'name': 'Mistral-7B',
        'hf_name': 'mistralai/Mistral-7B-v0.1',
        'use_4bit': True,
        'color': '#1f77b4',
        # Where existing human results are cached (zh, tr only — not en)
        'human_cache_dir': 'Wiki_multilingual_finegrain',
        'human_cache_pattern': '{lang}_intact_v1.json',
        'human_shuf_cache_pattern': '{lang}_shuffled_v1.json',
    },
    'llama': {
        'name': 'Llama-3-8B',
        'hf_name': 'unsloth/Meta-Llama-3.1-8B',
        'use_4bit': True,
        'color': '#9467bd',
        'human_cache_dir': 'Llama_crosslingual',
        'human_cache_pattern': 'wiki_{lang}_intact.json',
        'human_shuf_cache_pattern': 'wiki_{lang}_shuffled.json',
    },
}

# Only 3 languages for this comparison
LANGUAGES = {'en': 'English', 'zh': 'Chinese', 'tr': 'Turkish'}

MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42

# Check available data
print("AI articles:")
for lang, name in LANGUAGES.items():
    p = AI_DATA_DIR / f'{lang}_ai_articles.jsonl'
    if p.exists():
        with open(p) as f: n = sum(1 for _ in f)
        print(f'  {name}: {n} AI articles')
    else:
        print(f'  {name}: NOT FOUND')

print("\nHuman articles:")
for lang, name in LANGUAGES.items():
    p = HUMAN_DATA_DIR / f'{lang}_articles.jsonl'
    if p.exists():
        with open(p) as f: n = sum(1 for _ in f)
        print(f'  {name}: {n} human articles')
    else:
        print(f'  {name}: NOT FOUND')

In [ ]:
# === Core functions ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p
    return None

print('Functions defined')

In [ ]:
# === Process BOTH human and AI articles with each probe ===

for probe_key, probe_info in PROBE_MODELS.items():
    print(f'\n{"="*70}')
    print(f'PROBE: {probe_info["name"]}')
    print(f'{"="*70}')

    # Check if everything is cached
    all_cached = True
    for lang in LANGUAGES:
        for pop in ['human', 'ai']:
            ip = BASE_DIR / f'{probe_key}_{lang}_{pop}_intact.json'
            sp = BASE_DIR / f'{probe_key}_{lang}_{pop}_shuffled.json'
            if not (ip.exists() and sp.exists()):
                if pop == 'human' and lang != 'en':
                    h_dir = HUMAN_RESULTS / probe_info['human_cache_dir']
                    h_ip = h_dir / probe_info['human_cache_pattern'].format(lang=lang)
                    h_sp = h_dir / probe_info['human_shuf_cache_pattern'].format(lang=lang)
                    if h_ip.exists() and h_sp.exists():
                        import shutil
                        shutil.copy2(h_ip, ip)
                        shutil.copy2(h_sp, sp)
                        print(f'  {LANGUAGES[lang]} human: copied from previous run')
                        continue
                all_cached = False
    
    if all_cached:
        print(f'  All cached, skipping model load')
        # Still print results from cache
        print(f'\n  {"Language":<12} {"Pop":<8} {"α":>8} {"r":>8}')
        print(f'  {"-"*40}')
        for lang, name in LANGUAGES.items():
            for pop in ['human', 'ai']:
                ip = BASE_DIR / f'{probe_key}_{lang}_{pop}_intact.json'
                sp = BASE_DIR / f'{probe_key}_{lang}_{pop}_shuffled.json'
                if not (ip.exists() and sp.exists()): continue
                with open(ip) as f: intact = json.load(f)
                with open(sp) as f: shuffled = json.load(f)
                if len(intact) < 5 or len(shuffled) < 5: continue
                ipc = compute_raw_ppl_curve(intact)
                spc = compute_raw_ppl_curve(shuffled)
                corr = -np.diff(ipc) - (-np.diff(spc))
                fit = fit_power_law(corr)
                if fit:
                    print(f'  {name:<12} {pop:<8} {fit[0]:>8.3f} {fit[1]:>8.3f}')
        continue

    # Load model
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(probe_info['hf_name'])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        probe_info['hf_name'], quantization_config=bnb_config, device_map='auto'
    )
    model.eval()
    print(f'  Model loaded on {device}')

    @torch.no_grad()
    def compute_ppl(token_ids, target_start, target_end):
        if target_start >= target_end - 1:
            return float('inf')
        input_ids = torch.tensor([token_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        total_loss = 0.0
        count = 0
        for i in range(target_start, target_end - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            total_loss += -log_probs[token_ids[i + 1]].item()
            count += 1
        del outputs, logits
        torch.cuda.empty_cache()
        return math.exp(total_loss / count) if count > 0 else float('inf')

    def process_document(doc, rng_shuf):
        full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
        n = len(full_ids)
        intact_curves, shuffled_curves = [], []
        for frac in TARGET_FRACTIONS:
            target_start = int(n * frac)
            target_end = min(target_start + TARGET_LEN, n)
            if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5:
                continue
            target_ids = full_ids[target_start:target_end]
            context_pool = list(full_ids[:target_start])
            max_ctx = min(MAX_CONTEXT, len(context_pool))
            if max_ctx < 10:
                continue
            # Intact
            ppls, ctx_lengths = [], []
            for ctx_len in range(1, max_ctx + 1):
                chunk = context_pool[-ctx_len:] + target_ids
                ppl = compute_ppl(chunk, ctx_len, ctx_len + len(target_ids))
                if not math.isinf(ppl):
                    ppls.append(ppl)
                    ctx_lengths.append(ctx_len)
            if len(ppls) >= 10:
                intact_curves.append({'ctx_lengths': ctx_lengths, 'ppls': ppls,
                                      'doc_id': doc.get('doc_id',''), 'target_frac': frac})
            # Shuffled
            context_shuf = list(context_pool)
            rng_shuf.shuffle(context_shuf)
            ppls_s, ctx_s = [], []
            for ctx_len in range(1, max_ctx + 1):
                chunk = context_shuf[-ctx_len:] + target_ids
                ppl = compute_ppl(chunk, ctx_len, ctx_len + len(target_ids))
                if not math.isinf(ppl):
                    ppls_s.append(ppl)
                    ctx_s.append(ctx_len)
            if len(ppls_s) >= 10:
                shuffled_curves.append({'ctx_lengths': ctx_s, 'ppls': ppls_s,
                                        'doc_id': doc.get('doc_id',''), 'target_frac': frac})
        return intact_curves, shuffled_curves

    # Track results for running summary
    probe_results = {}

    for lang, name in LANGUAGES.items():
        for pop, data_dir, file_pattern in [
            ('human', HUMAN_DATA_DIR, f'{lang}_articles.jsonl'),
            ('ai', AI_DATA_DIR, f'{lang}_ai_articles.jsonl'),
        ]:
            intact_path = BASE_DIR / f'{probe_key}_{lang}_{pop}_intact.json'
            shuffled_path = BASE_DIR / f'{probe_key}_{lang}_{pop}_shuffled.json'

            if intact_path.exists() and shuffled_path.exists():
                with open(intact_path) as f: intact = json.load(f)
                with open(shuffled_path) as f: shuffled = json.load(f)
                print(f'  {name} {pop}: cached ({len(intact)} curves)')
            else:
                data_path = data_dir / file_pattern
                if not data_path.exists():
                    print(f'  {name} {pop}: NO DATA at {data_path}')
                    continue

                corpus = []
                with open(data_path) as f:
                    for line in f:
                        corpus.append(json.loads(line))
                print(f'  {name} {pop}: processing {len(corpus)} articles...')

                intact, shuffled = [], []
                rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
                for doc in tqdm(corpus, desc=f'{name} {pop}'):
                    i, s = process_document(doc, rng_shuf)
                    intact.extend(i)
                    shuffled.extend(s)

                with open(intact_path, 'w') as f:
                    json.dump(intact, f)
                with open(shuffled_path, 'w') as f:
                    json.dump(shuffled, f)

            # Fit and report immediately
            fit = None
            if len(intact) >= 5 and len(shuffled) >= 5:
                ipc = compute_raw_ppl_curve(intact)
                spc = compute_raw_ppl_curve(shuffled)
                corr = -np.diff(ipc) - (-np.diff(spc))
                fit = fit_power_law(corr)
            if fit:
                print(f'    >> {name} {pop}: α = {fit[0]:.3f} (r={fit[1]:.3f}, n={len(intact)})')
                probe_results[(lang, pop)] = fit[0]
            else:
                print(f'    >> {name} {pop}: fit failed')

        # After each language, print running comparison if we have both
        if (lang, 'human') in probe_results and (lang, 'ai') in probe_results:
            h_alpha = probe_results[(lang, 'human')]
            a_alpha = probe_results[(lang, 'ai')]
            delta = a_alpha - h_alpha
            direction = 'AI STEEPER' if a_alpha < h_alpha else 'AI SHALLOWER' if a_alpha > h_alpha else 'SAME'
            print(f'\n    *** {name}: human={h_alpha:.3f}, AI={a_alpha:.3f}, Δ={delta:.3f} ({direction}) ***\n')

    # Probe summary
    print(f'\n  --- {probe_info["name"]} SUMMARY ---')
    for lang, name in LANGUAGES.items():
        if (lang, 'human') in probe_results and (lang, 'ai') in probe_results:
            print(f'  {name:<12} human={probe_results[(lang,"human")]:.3f}  AI={probe_results[(lang,"ai")]:.3f}  Δ={probe_results[(lang,"ai")]-probe_results[(lang,"human")]:.3f}')

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'  Model unloaded')

print('\nAll done!')

In [ ]:
# === Compare human vs AI ===

print(f'{"Probe":<12} {"Language":<12} {"Human α":>10} {"AI α":>10} {"Human r":>10} {"AI r":>10} {"Δα":>8}')
print('-' * 75)

all_human_exps = []
all_ai_exps = []
comparison_rows = []

for probe_key, probe_info in PROBE_MODELS.items():
    for lang, name in LANGUAGES.items():
        results = {}
        for pop in ['human', 'ai']:
            ip = BASE_DIR / f'{probe_key}_{lang}_{pop}_intact.json'
            sp = BASE_DIR / f'{probe_key}_{lang}_{pop}_shuffled.json'
            if not (ip.exists() and sp.exists()):
                continue
            with open(ip) as f: intact = json.load(f)
            with open(sp) as f: shuffled = json.load(f)
            if len(intact) < 5 or len(shuffled) < 5:
                continue
            ipc = compute_raw_ppl_curve(intact)
            spc = compute_raw_ppl_curve(shuffled)
            corr = -np.diff(ipc) - (-np.diff(spc))
            fit = fit_power_law(corr)
            if fit:
                results[pop] = {'slope': fit[0], 'r': fit[1], 'p': fit[2], 'n': len(intact)}

        if 'human' in results and 'ai' in results:
            h = results['human']
            a = results['ai']
            delta = a['slope'] - h['slope']
            print(f'{probe_info["name"]:<12} {name:<12} {h["slope"]:>10.3f} {a["slope"]:>10.3f} '
                  f'{h["r"]:>10.3f} {a["r"]:>10.3f} {delta:>8.3f}')
            all_human_exps.append(h['slope'])
            all_ai_exps.append(a['slope'])
            comparison_rows.append({
                'probe': probe_info['name'], 'language': name,
                'human_alpha': h['slope'], 'ai_alpha': a['slope'],
                'human_r': h['r'], 'ai_r': a['r'], 'delta': delta,
            })

if all_human_exps and all_ai_exps:
    print(f'\nHuman mean: {np.mean(all_human_exps):.3f} ± {np.std(all_human_exps):.3f}')
    print(f'AI mean:    {np.mean(all_ai_exps):.3f} ± {np.std(all_ai_exps):.3f}')
    if len(all_human_exps) >= 3:
        t, p = stats.ttest_rel(all_human_exps, all_ai_exps)
        print(f'Paired t-test: t={t:.3f}, p={p:.4f}')
    # Direction check
    n_steeper = sum(1 for h, a in zip(all_human_exps, all_ai_exps) if a < h)
    print(f'AI steeper than human: {n_steeper}/{len(all_human_exps)} comparisons')

In [ ]:
# === Figure ===
if comparison_rows:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    labels = [f'{r["language"]}\n({r["probe"]})' for r in comparison_rows]
    h_vals = [r['human_alpha'] for r in comparison_rows]
    a_vals = [r['ai_alpha'] for r in comparison_rows]

    # Panel A: Paired bar chart
    ax = axes[0]
    x = np.arange(len(labels))
    w = 0.35
    ax.bar(x - w/2, h_vals, w, color='#2196F3', alpha=0.7, label='Human', edgecolor='black')
    ax.bar(x + w/2, a_vals, w, color='#f44336', alpha=0.7, label='AI (Claude)', edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel('Power Law Exponent (α)')
    ax.set_title('Human vs AI Wikipedia: Matched Topics', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2, axis='y')

    # Panel B: Scatter
    ax = axes[1]
    ax.scatter(h_vals, a_vals, s=80, c='purple', edgecolors='black', zorder=5)
    for i, r in enumerate(comparison_rows):
        ax.annotate(f'{r["language"]}\n({r["probe"]})', (h_vals[i], a_vals[i]),
                    fontsize=6, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
    all_vals = h_vals + a_vals
    lims = [min(all_vals) - 0.15, max(all_vals) + 0.15]
    ax.plot(lims, lims, '--', color='gray', alpha=0.5, label='y=x (identical)')
    ax.set_xlabel('Human exponent (α)')
    ax.set_ylabel('AI exponent (α)')
    ax.set_title('Human vs AI (paired)', fontweight='bold')
    if len(h_vals) >= 3:
        r_corr, p_corr = stats.pearsonr(h_vals, a_vals)
        ax.legend(title=f'r={r_corr:.2f}, p={p_corr:.3f}', fontsize=9)
    else:
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    plt.suptitle('Human vs AI-Generated Wikipedia: Coherence Decay',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig1_human_vs_ai.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No comparison data available')